# Notebook 02 — RibFrac CT Preprocessing

This notebook preprocesses RibFrac CT volumes and fracture annotation volumes for the Enhanced 3D ResUNet framework.

**Architecture:** preprocessing is performed in Kaggle `/kaggle/working`, then the resulting `ribfrac_preprocessed` folder can be published as a Kaggle Dataset. Notebook 03 will consume that persistent Dataset rather than depending on Notebook 02 running simultaneously.

The single-patient pipeline is verified first using `RibFrac128`. Full training-set preprocessing is disabled until this verification succeeds.


In [ ]:
# Cell 1 — Import libraries

import os
import gc
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")


In [ ]:
# Cell 2 — Reproducibility

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print("Random seed:", SEED)


In [ ]:
# Cell 3 — Clone/load GitHub helper module

import subprocess

REPO_URL = "https://github.com/waboke/3D_ResUnet_Zenodo.git"
REPO_DIR = Path("/kaggle/working/3D_ResUnet_Zenodo")

if not REPO_DIR.exists():
    print("Cloning GitHub repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    print("GitHub repository already exists.")

sys.path.insert(0, str(REPO_DIR))

from ribfrac_data_zenodo import (
    download_ribfrac_image,
    get_label_path,
    load_patient,
    validate_patient,
)

print("RibFrac helper module loaded successfully.")


In [ ]:
# Cell 4 — Directory configuration

BASE_DIR = Path("/kaggle/working")

IMAGE_DIR = BASE_DIR / "ribfrac_images"
LABEL_DIR = BASE_DIR / "ribfrac_labels"
METADATA_DIR = BASE_DIR / "ribfrac_metadata"
INFO_PATH = METADATA_DIR / "ribfrac-train-info-1.csv"

# Final output that will later be published as a Kaggle Dataset
PREPROCESSED_DIR = BASE_DIR / "ribfrac_preprocessed"
PREPROCESSED_IMAGE_DIR = PREPROCESSED_DIR / "images"
PREPROCESSED_LABEL_DIR = PREPROCESSED_DIR / "labels"
PREPROCESSED_METADATA_DIR = PREPROCESSED_DIR / "metadata"

for directory in [
    IMAGE_DIR,
    LABEL_DIR,
    METADATA_DIR,
    PREPROCESSED_IMAGE_DIR,
    PREPROCESSED_LABEL_DIR,
    PREPROCESSED_METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Preprocessed output root:", PREPROCESSED_DIR)
print("Images:", PREPROCESSED_IMAGE_DIR)
print("Labels:", PREPROCESSED_LABEL_DIR)
print("Metadata:", PREPROCESSED_METADATA_DIR)


In [ ]:
# Cell 5 — Download/verify metadata

INFO_URL = (
    "https://zenodo.org/records/3893508/"
    "files/ribfrac-train-info-1.csv"
)

if not INFO_PATH.exists():

    import requests

    print("Downloading RibFrac metadata...")

    response = requests.get(
        INFO_URL,
        timeout=120
    )

    response.raise_for_status()

    INFO_PATH.write_bytes(
        response.content
    )

    print("Metadata downloaded.")

else:

    print("Metadata already exists.")

print("Metadata path:", INFO_PATH)
print("File size:", INFO_PATH.stat().st_size, "bytes")


In [ ]:
# Cell 6 — Load metadata

info_df = pd.read_csv(INFO_PATH)

print("Metadata shape:", info_df.shape)
print("Columns:", info_df.columns.tolist())

display(info_df.head())


In [ ]:
# Cell 7 — Patient-level split

patient_ids = sorted(
    info_df["public_id"].unique()
)

train_patients, test_patients = train_test_split(
    patient_ids,
    test_size=0.20,
    random_state=SEED
)

train_patients, val_patients = train_test_split(
    train_patients,
    test_size=0.20,
    random_state=SEED
)

print("Unique patients:", len(patient_ids))
print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

assert not (set(train_patients) & set(val_patients))
assert not (set(train_patients) & set(test_patients))
assert not (set(val_patients) & set(test_patients))

print("Patient-level split integrity: PASS")


In [ ]:
# Cell 8 — Save patient split

train_set = set(train_patients)
val_set = set(val_patients)

split_df = pd.DataFrame({
    "public_id": patient_ids,
    "split": [
        "train" if p in train_set
        else "validation" if p in val_set
        else "test"
        for p in patient_ids
    ]
})

split_path = (
    PREPROCESSED_METADATA_DIR /
    "patient_split.csv"
)

split_df.to_csv(
    split_path,
    index=False
)

print("Patient split saved:", split_path)

display(
    split_df["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="patients")
)


In [ ]:
# Cell 9 — Preprocessing parameters

TARGET_SPACING = (1.0, 1.0, 1.0)

HU_MIN = -1000.0
HU_MAX = 400.0

print("Target spacing:", TARGET_SPACING)
print("HU clipping range:", (HU_MIN, HU_MAX))


In [ ]:
# Cell 10 — Load one patient

def load_patient_for_preprocessing(patient_id):

    ct_path = download_ribfrac_image(
        patient_id
    )

    label_path = get_label_path(
        patient_id
    )

    ct_img = nib.load(
        str(ct_path)
    )

    label_img = nib.load(
        str(label_path)
    )

    ct_volume = ct_img.get_fdata(
        dtype=np.float32
    )

    label_volume = (
        label_img
        .get_fdata()
        .astype(np.int16)
    )

    return {
        "ct_img": ct_img,
        "label_img": label_img,
        "ct_volume": ct_volume,
        "label_volume": label_volume,
    }

print("Patient loading function defined.")


In [ ]:
# Cell 11 — CT intensity preprocessing

def preprocess_ct_intensity(
    ct_volume
):

    ct = np.clip(
        ct_volume,
        HU_MIN,
        HU_MAX
    )

    ct = (
        ct - HU_MIN
    ) / (
        HU_MAX - HU_MIN
    )

    return ct.astype(
        np.float32
    )

print("CT intensity preprocessing function defined.")


In [ ]:
# Cell 12 — Resampling functions

def calculate_zoom_factors(
    original_spacing,
    target_spacing
):

    return tuple(
        float(original) / float(target)
        for original, target
        in zip(
            original_spacing,
            target_spacing
        )
    )


def resample_ct(
    ct_volume,
    original_spacing,
    target_spacing
):

    zoom_factors = (
        calculate_zoom_factors(
            original_spacing,
            target_spacing
        )
    )

    return zoom(
        ct_volume,
        zoom=zoom_factors,
        order=1
    ).astype(
        np.float32
    )


def resample_label(
    label_volume,
    original_spacing,
    target_spacing
):

    zoom_factors = (
        calculate_zoom_factors(
            original_spacing,
            target_spacing
        )
    )

    return zoom(
        label_volume,
        zoom=zoom_factors,
        order=0
    ).astype(
        np.int16
    )

print("Resampling functions defined.")


In [ ]:
# Cell 13 — Complete preprocessing function

def preprocess_patient(
    patient_id
):

    print("=" * 60)
    print(
        f"Processing patient: {patient_id}"
    )
    print("=" * 60)

    patient = (
        load_patient_for_preprocessing(
            patient_id
        )
    )

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct_volume = patient["ct_volume"]
    label_volume = patient["label_volume"]

    validation = validate_patient(
        ct_img,
        label_img
    )

    if not validation["shape_match"]:
        raise ValueError(
            f"CT/label shape mismatch for {patient_id}"
        )

    if not validation["spacing_match"]:
        raise ValueError(
            f"CT/label spacing mismatch for {patient_id}"
        )

    if not validation["affine_match"]:
        raise ValueError(
            f"CT/label affine mismatch for {patient_id}"
        )

    print("Spatial validation: PASS")

    original_spacing = (
        ct_img.header.get_zooms()[:3]
    )

    print(
        "Original spacing:",
        original_spacing
    )

    ct_volume = preprocess_ct_intensity(
        ct_volume
    )

    print(
        "CT intensity range after normalization:",
        float(ct_volume.min()),
        "to",
        float(ct_volume.max())
    )

    ct_resampled = resample_ct(
        ct_volume,
        original_spacing,
        TARGET_SPACING
    )

    label_resampled = resample_label(
        label_volume,
        original_spacing,
        TARGET_SPACING
    )

    if (
        ct_resampled.shape
        != label_resampled.shape
    ):
        raise ValueError(
            f"Resampled CT/label shape mismatch "
            f"for {patient_id}"
        )

    unique_labels = np.unique(
        label_resampled
    )

    if not np.all(
        unique_labels
        == unique_labels.astype(int)
    ):
        raise ValueError(
            f"Non-integer label values detected "
            f"for {patient_id}"
        )

    print(
        "Resampled CT shape:",
        ct_resampled.shape
    )

    print(
        "Resampled label shape:",
        label_resampled.shape
    )

    print(
        "CT range:",
        float(ct_resampled.min()),
        "to",
        float(ct_resampled.max())
    )

    print(
        "Label values:",
        unique_labels
    )

    return {
        "patient_id": patient_id,
        "ct": ct_resampled,
        "label": label_resampled,
        "original_spacing": tuple(
            original_spacing
        ),
        "target_spacing": TARGET_SPACING,
    }

print("Complete preprocessing function defined.")


In [ ]:
# Cell 14 — Save preprocessed patient

def save_preprocessed_patient(
    patient_data
):

    patient_id = (
        patient_data["patient_id"]
    )

    ct = patient_data["ct"]
    label = patient_data["label"]

    ct_path = (
        PREPROCESSED_IMAGE_DIR /
        f"{patient_id}-image.npy"
    )

    label_path = (
        PREPROCESSED_LABEL_DIR /
        f"{patient_id}-label.npy"
    )

    np.save(
        ct_path,
        ct.astype(np.float32)
    )

    np.save(
        label_path,
        label.astype(np.int16)
    )

    print(
        "Saved CT:",
        ct_path
    )

    print(
        "Saved label:",
        label_path
    )

    return ct_path, label_path

print("Save function defined.")


## RibFrac128 validation

Run the following cells first. Full training-set preprocessing remains disabled until these checks pass.


In [ ]:
# Cell 15 — Preprocess and save RibFrac128

TEST_PATIENT_ID = "RibFrac128"

test_patient = preprocess_patient(
    TEST_PATIENT_ID
)

ct_path, label_path = (
    save_preprocessed_patient(
        test_patient
    )
)

print("RibFrac128 preprocessing completed.")


In [ ]:
# Cell 16 — Verify saved RibFrac128 files

saved_ct = np.load(
    ct_path
)

saved_label = np.load(
    label_path
)

print("Saved CT shape:", saved_ct.shape)
print("Saved label shape:", saved_label.shape)

print(
    "Saved CT dtype:",
    saved_ct.dtype
)

print(
    "Saved label dtype:",
    saved_label.dtype
)

print(
    "Saved CT range:",
    float(saved_ct.min()),
    "to",
    float(saved_ct.max())
)

print(
    "Saved label values:",
    np.unique(saved_label)
)

assert (
    saved_ct.shape
    == saved_label.shape
)

assert saved_ct.dtype == np.float32
assert saved_label.dtype == np.int16

assert np.isfinite(
    saved_ct
).all()

assert (
    np.all(saved_ct >= 0.0)
    and np.all(saved_ct <= 1.0)
)

unique_saved_labels = np.unique(
    saved_label
)

assert np.all(
    unique_saved_labels
    == unique_saved_labels.astype(int)
)

print()
print(
    "RibFrac128 saved-file verification: PASS"
)


In [ ]:
# Cell 17 — Release test arrays

del test_patient
del saved_ct
del saved_label

gc.collect()

print(
    "Temporary RibFrac128 arrays released."
)


## Full training-set preprocessing

The following cell is intentionally disabled.

After RibFrac128 passes all checks, set `PROCESS_TRAINING_SET = True` and run it once to preprocess the 192 development-training patients sequentially.


In [ ]:
# Cell 18 — Optional full training-set preprocessing

PROCESS_TRAINING_SET = False

if PROCESS_TRAINING_SET:

    for i, patient_id in enumerate(
        train_patients,
        start=1
    ):

        print(
            f"\nPatient {i}/{len(train_patients)}"
        )

        print(
            f"Patient ID: {patient_id}"
        )

        processed = preprocess_patient(
            patient_id
        )

        save_preprocessed_patient(
            processed
        )

        del processed

        gc.collect()

    print(
        "\nTraining-set preprocessing completed."
    )

else:

    print(
        "Training-set preprocessing is DISABLED."
    )

    print(
        "Set PROCESS_TRAINING_SET = True "
        "only after RibFrac128 verification passes."
    )


In [ ]:
# Cell 19 — Final output verification

image_files = sorted(
    PREPROCESSED_IMAGE_DIR.glob(
        "*.npy"
    )
)

label_files = sorted(
    PREPROCESSED_LABEL_DIR.glob(
        "*.npy"
    )
)

print(
    "Preprocessed output root:",
    PREPROCESSED_DIR
)

print(
    "CT files:",
    len(image_files)
)

print(
    "Label files:",
    len(label_files)
)

print()
print("Output files:")

for path in sorted(
    PREPROCESSED_DIR.rglob("*")
):

    if path.is_file():

        print(
            path.relative_to(
                PREPROCESSED_DIR
            )
        )


In [ ]:
# Cell 20 — Save preprocessing configuration

config = {
    "notebook": "02_Preprocessing",
    "seed": SEED,
    "target_spacing": list(
        TARGET_SPACING
    ),
    "hu_clip_min": HU_MIN,
    "hu_clip_max": HU_MAX,
    "ct_interpolation_order": 1,
    "label_interpolation_order": 0,
    "test_patient": TEST_PATIENT_ID,
    "training_set_processed": (
        PROCESS_TRAINING_SET
    ),
    "training_patients": len(
        train_patients
    ),
    "validation_patients": len(
        val_patients
    ),
    "test_patients": len(
        test_patients
    ),
}

config_path = (
    PREPROCESSED_METADATA_DIR /
    "preprocessing_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

print(
    "Configuration saved:",
    config_path
)


## Final status

The folder `/kaggle/working/ribfrac_preprocessed/` is the persistent artifact produced by this notebook.

After the required patients have been processed and verified, publish this folder as a Kaggle Dataset, for example:

`ribfrac-preprocessed`

Notebook 03 will then use that Dataset as its input.

**Important:** Notebook 03 should not read Notebook 02's `/kaggle/working` directory. It should read the published Kaggle Dataset under `/kaggle/input/...`.
